In [1]:
!pip install -U \
langchain==1.3.7 \
langchain-classic \
langchain-community \
langchain-core \
langchain-text-splitters \
langchain-huggingface \
langchain-openai \
sentence-transformers \
faiss-cpu \
pypdf \
chromadb \
tiktoken

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.4/131.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.3/554.3 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores import Chroma

from langchain_openai import ChatOpenAI

from langchain_classic.chains import RetrievalQA

/tmp/ipykernel_894/3815517231.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [9]:
from google.colab import files

uploaded = files.upload()

Saving computer_networks.txt to computer_networks.txt
Saving machine_learning.txt to machine_learning.txt
Saving operating_systems.txt to operating_systems.txt
Saving web_developement.txt to web_developement.txt


In [11]:
files_list = [
    "operating_systems.txt",
    "machine_learning.txt",
    "computer_networks.txt",
    "web_developement.txt"
]

documents = []

for file in files_list:
    loader = TextLoader(file, encoding="utf-8")
    documents.extend(loader.load())

print("Documents Loaded:", len(documents))

Documents Loaded: 4


In [12]:
splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

texts = splitter.split_documents(documents)

print("Chunks:", len(texts))

Chunks: 4


In [13]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
vectorstore = FAISS.from_documents(
    texts,
    embeddings
)

print("Vector Database Created")

Vector Database Created


In [15]:

retriever = vectorstore.as_retriever()

docs = retriever.invoke(
    "What is FCFS Scheduling?"
)

print(docs[0].page_content)

Operating Systems (OS) is system software that manages computer hardware and software resources.

Key Functions:
- Process Management: Handles creation, scheduling, and termination of processes.
- Memory Management: Allocates RAM efficiently.
- File System Management: Organizes and stores files.
- Device Management: Controls input/output devices.

Types of Operating Systems:
- Batch OS
- Time-Sharing OS
- Distributed OS
- Real-Time OS

Popular Operating Systems:
- Windows
- Linux
- macOS

Concepts:
- Multithreading
- Virtual Memory
- Deadlock
- Scheduling Algorithms (FCFS, SJF, Round Robin)


In [16]:
!pip install -q langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.0 MB/s eta 0:00:00


In [17]:
from langchain_groq import ChatGroq
import os

In [23]:
import getpass

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter Groq API Key:")

KeyboardInterrupt: Interrupted by user

In [19]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2
)

In [20]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    chain_type="stuff"
)

In [21]:
response = qa.invoke({
    "query": "What is FCFS Scheduling?"
})

print(response["result"])

FCFS stands for First-Come-First-Served. It is a scheduling algorithm used in Operating Systems for process management. 

In FCFS scheduling, the process that arrives first in the ready queue is executed first by the CPU. The next process in the queue is executed only after the previous process has completed its execution or has been terminated.

The key characteristics of FCFS scheduling are:

- It is a non-preemptive scheduling algorithm, meaning that once a process starts execution, it cannot be interrupted by another process until it completes.
- It is a simple and easy-to-implement algorithm.
- It can lead to poor system performance if a long-running process is executed first, causing other processes to wait for a long time.

FCFS scheduling is often used in batch processing systems where processes are executed in the order they are received.


In [22]:
while True:
    question = input("Ask Question: ")

    if question.lower() == "exit":
        break

    response = qa.invoke({
        "query": question
    })

    print("\nAnswer:")
    print(response["result"])
    print("-" * 50)

Ask Question: what is os

Answer:
An Operating System (OS) is a type of system software that manages computer hardware and software resources. Its key functions include:

1. Process Management: Handles creation, scheduling, and termination of processes.
2. Memory Management: Allocates RAM efficiently.
3. File System Management: Organizes and stores files.
4. Device Management: Controls input/output devices.

In other words, an OS acts as an intermediary between computer hardware and user-level applications, allowing users to interact with the computer and its resources in a convenient and efficient way. Examples of popular Operating Systems include Windows, Linux, and macOS.
--------------------------------------------------
Ask Question: who is racchith

Answer:
I don't know who Racchith is. The provided context does not mention anyone by that name. If you could provide more information or context about Racchith, I may be able to help you better.
--------------------------------------

KeyboardInterrupt: Interrupted by user